# RAG Operations Notebook

Plain notebook workflow for:
- fetching source data,
- building collections,
- refreshing aliases,
- inspecting collection state,
- and doing direct Qdrant repair when necessary.

## Notebook Layout
1. Imports
2. Setup & current state
3. Fetch source data for ArXiv
4. Fetch source data for PyTorch docs
5. Bootstrap new collections
6. Refresh existing aliases
7. Alias management
8. Diagnostics
9. Direct Qdrant exploration
10. Raw Qdrant management

## Notes
- Each write cell performs one operation only.
- Edit values only in the cell you plan to run.
- Source-data fetching reuses the same Python helpers as the Airflow DAGs via `shared.rag_data_fetchers`.
- Production-safe collection work goes through `src/rag/ops`.
- Direct Qdrant cells are for investigation or manual repair only.

## 1. Imports

In [ ]:
import json
import sys
from pathlib import Path
from pprint import pprint

for candidate in (
    Path("/home/jovyan"),
    Path("/home/jovyan/src"),
    Path("/home/anton-m/Git/agent-042"),
    Path("/home/anton-m/Git/agent-042/src"),
):
    if candidate.exists():
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

from rag.ops import (
    BuildConfig,
    ImplementationInfo,
    assign_alias_to_collection,
    create_arxiv_collection,
    create_pytorch_docs_collection,
    detach_alias,
    inspect_alias,
    inspect_collection,
    list_alias_mappings,
    promote_alias,
    update_arxiv_collection,
    update_pytorch_docs_collection,
)
from rag.vector_store import QdrantVectorStore
from shared.config import bootstrap_local_settings_env, get_knowledge_bases, get_settings

try:
    from shared.rag_data_fetchers import (
        DEFAULT_ARXIV_CATEGORIES,
        DEFAULT_ARXIV_DELAY_SECONDS,
        DEFAULT_ARXIV_PAGE_SIZE,
        DEFAULT_PYTORCH_BASE_URL,
        DEFAULT_PYTORCH_MAX_CODE_EXAMPLES,
        DEFAULT_PYTORCH_PAGES,
        DEFAULT_PYTORCH_SCRAPE_DELAY_SECONDS,
        build_arxiv_query,
        collect_pytorch_docs,
        download_arxiv_papers,
        get_arxiv_total_results,
    )
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "shared.rag_data_fetchers is missing from the Jupyter server checkout under "
        "/home/jovyan/src/shared. Pull the latest repo code on the server before running this notebook."
    ) from exc

RuntimeError: shared.rag_data_fetchers is missing from the Jupyter server checkout under /home/jovyan/src/shared. Pull the latest repo code on the server before running this notebook.

## 2. Setup & Current State

This cell wires the notebook to the repo on the remote server, loads settings, and shows the current knowledge-base and alias state.

In [ ]:
repo_root = Path("/home/jovyan")
# repo_root = Path("/home/anton-m/Git/agent-042")

assert repo_root.exists(), f"repo_root does not exist: {repo_root}"

loaded_env = bootstrap_local_settings_env(repo_root=repo_root)
settings = get_settings()

ARXIV_DATA_FILE = repo_root / "assets" / "rag_data" / "arxiv" / "arxiv_papers.json"
PYTORCH_DOCS_FILE = repo_root / "assets" / "rag_data" / "pytorch_docs" / "pytorch_docs.json"

kb_registry = {
    task_name: [
        {
            "name": kb.name,
            "aliases": kb.aliases,
            "update_strategy": kb.update_strategy,
            "label": kb.label,
        }
        for kb in task_cfg.knowledge_bases
    ]
    for task_name, task_cfg in get_knowledge_bases().items()
}

pprint(
    {
        "repo_root": str(repo_root),
        "loaded_env": loaded_env,
        "qdrant_endpoint": f"{settings.qdrant_host}:{settings.qdrant_port}",
        "arxiv_data_file": {"exists": ARXIV_DATA_FILE.exists(), "path": str(ARXIV_DATA_FILE)},
        "pytorch_docs_file": {
            "exists": PYTORCH_DOCS_FILE.exists(),
            "path": str(PYTORCH_DOCS_FILE),
        },
        "knowledge_bases": kb_registry,
        "live_aliases": list_alias_mappings(),
    }
)

## 3. Fetch Source Data — ArXiv

These cells reuse `shared.rag_data_fetchers`, the same helper module used by the ArXiv Airflow DAG.
Start with the preview cell to check the current match count before you download anything.

In [ ]:
ARXIV_FETCH = {
    "categories": list(DEFAULT_ARXIV_CATEGORIES),
    "max_results": 2000,
    "page_size": DEFAULT_ARXIV_PAGE_SIZE,
    "delay_seconds": DEFAULT_ARXIV_DELAY_SECONDS,
    "max_retries": 5,
    "request_timeout": 60,
    "output_dir": repo_root / "assets" / "rag_data" / "arxiv",
}

arxiv_query = build_arxiv_query(ARXIV_FETCH["categories"])
arxiv_total_results = get_arxiv_total_results(
    ARXIV_FETCH["categories"],
    request_timeout=ARXIV_FETCH["request_timeout"],
    max_retries=ARXIV_FETCH["max_retries"],
    base_delay_seconds=ARXIV_FETCH["delay_seconds"],
    user_agent="agent-042-rag-ops-notebook/1.0",
)

pprint(
    {
        "query": arxiv_query,
        "categories": ARXIV_FETCH["categories"],
        "total_results": arxiv_total_results,
        "download_plan": {
            "max_results": ARXIV_FETCH["max_results"],
            "page_size": ARXIV_FETCH["page_size"],
            "delay_seconds": ARXIV_FETCH["delay_seconds"],
            "output_dir": str(ARXIV_FETCH["output_dir"]),
        },
    }
)

### 3.1 Download ArXiv Metadata

In [ ]:
arxiv_fetch_summary = download_arxiv_papers(
    categories=ARXIV_FETCH["categories"],
    max_results=ARXIV_FETCH["max_results"],
    output_dir=ARXIV_FETCH["output_dir"],
    page_size=ARXIV_FETCH["page_size"],
    delay_seconds=ARXIV_FETCH["delay_seconds"],
    max_retries=ARXIV_FETCH["max_retries"],
    request_timeout=ARXIV_FETCH["request_timeout"],
    user_agent="agent-042-rag-ops-notebook/1.0",
)

pprint(arxiv_fetch_summary)

### 3.2 Inspect ArXiv Dataset

In [ ]:
if not ARXIV_DATA_FILE.exists():
    print(f"File not found: {ARXIV_DATA_FILE}")
else:
    with open(ARXIV_DATA_FILE, encoding="utf-8") as file_handle:
        arxiv_papers = json.load(file_handle)

    print(f"Total papers: {len(arxiv_papers)}")
    if arxiv_papers:
        pprint(arxiv_papers[0])
    else:
        print("Dataset is empty.")

### 3.3 Bulk ArXiv Metadata — OAI-PMH

For large metadata pulls, prefer arXiv's official OAI-PMH interface instead of the search API.

- Base URL: `https://oaipmh.arxiv.org/oai`
- Best use: full metadata harvests and incremental syncs.
- Caveat: OAI-PMH datestamps track record updates, not original submission dates.
- The default set list below matches the ML/AI categories used elsewhere in this project.
- For full-text PDFs or source tarballs, use arXiv's official requester-pays S3 bucket; the manifest locations are shown in the config cell.

In [ ]:
from shared.rag_data_fetchers import (
    DEFAULT_ARXIV_OAI_BASE_URL,
    DEFAULT_ARXIV_OAI_DELAY_SECONDS,
    DEFAULT_ARXIV_OAI_METADATA_PREFIX,
    DEFAULT_ARXIV_OAI_SET_SPECS,
    DEFAULT_ARXIV_S3_BULK_DOCS_URL,
    DEFAULT_ARXIV_S3_PDF_MANIFEST_URL,
    DEFAULT_ARXIV_S3_SOURCE_MANIFEST_URL,
)

ARXIV_BULK = {
    "base_url": DEFAULT_ARXIV_OAI_BASE_URL,
    "metadata_prefix": DEFAULT_ARXIV_OAI_METADATA_PREFIX,
    "set_specs": list(DEFAULT_ARXIV_OAI_SET_SPECS),
    "output_dir": repo_root / "assets" / "rag_data" / "arxiv_bulk_oai",
    "delay_seconds": DEFAULT_ARXIV_OAI_DELAY_SECONDS,
    "max_pages_per_set": 2,  # Set to None for a full harvest.
    "request_timeout": 60,
    "max_retries": 5,
}

pprint(
    {
        "metadata_harvest": {
            "base_url": ARXIV_BULK["base_url"],
            "metadata_prefix": ARXIV_BULK["metadata_prefix"],
            "set_specs": ARXIV_BULK["set_specs"],
            "output_dir": str(ARXIV_BULK["output_dir"]),
            "delay_seconds": ARXIV_BULK["delay_seconds"],
            "max_pages_per_set": ARXIV_BULK["max_pages_per_set"],
            "request_timeout": ARXIV_BULK["request_timeout"],
            "max_retries": ARXIV_BULK["max_retries"],
        },
        "official_full_text": {
            "docs_url": DEFAULT_ARXIV_S3_BULK_DOCS_URL,
            "pdf_manifest_url": DEFAULT_ARXIV_S3_PDF_MANIFEST_URL,
            "source_manifest_url": DEFAULT_ARXIV_S3_SOURCE_MANIFEST_URL,
        },
    }
)

In [ ]:
from shared.rag_data_fetchers import harvest_arxiv_metadata_oai

arxiv_bulk_summary = harvest_arxiv_metadata_oai(
    set_specs=ARXIV_BULK["set_specs"],
    output_dir=ARXIV_BULK["output_dir"],
    base_url=ARXIV_BULK["base_url"],
    metadata_prefix=ARXIV_BULK["metadata_prefix"],
    delay_seconds=ARXIV_BULK["delay_seconds"],
    request_timeout=ARXIV_BULK["request_timeout"],
    max_retries=ARXIV_BULK["max_retries"],
    max_pages_per_set=ARXIV_BULK["max_pages_per_set"],
    user_agent="agent-042-rag-ops-notebook/1.0",
)

pprint(arxiv_bulk_summary)

In [ ]:
if "arxiv_bulk_summary" not in globals():
    print("Run the bulk harvest cell first.")
else:
    summary_file = Path(arxiv_bulk_summary["summary_file"])
    with open(summary_file, encoding="utf-8") as file_handle:
        bulk_summary = json.load(file_handle)

    pprint(
        {
            "total_records": bulk_summary["total_records"],
            "total_deleted_records": bulk_summary["total_deleted_records"],
            "sets": bulk_summary["sets"],
        }
    )

    first_output_file = (
        Path(bulk_summary["sets"][0]["output_file"]) if bulk_summary["sets"] else None
    )
    if first_output_file is None:
        print("No set outputs were produced.")
    else:
        with open(first_output_file, encoding="utf-8") as file_handle:
            first_line = file_handle.readline().strip()

        if first_line:
            pprint(json.loads(first_line))
        else:
            print(f"No records found in {first_output_file}")

## 4. Fetch Source Data — PyTorch Docs

This cell reuses `shared.rag_data_fetchers.collect_pytorch_docs`, which is the same helper used by the PyTorch docs DAG.

In [ ]:
PYTORCH_FETCH = {
    "base_url": DEFAULT_PYTORCH_BASE_URL,
    "page_list": list(DEFAULT_PYTORCH_PAGES),
    "delay_seconds": DEFAULT_PYTORCH_SCRAPE_DELAY_SECONDS,
    "max_code_examples": DEFAULT_PYTORCH_MAX_CODE_EXAMPLES,
    "output_dir": repo_root / "assets" / "rag_data" / "pytorch_docs",
}

pytorch_fetch_summary = collect_pytorch_docs(
    base_url=PYTORCH_FETCH["base_url"],
    page_list=PYTORCH_FETCH["page_list"],
    output_dir=PYTORCH_FETCH["output_dir"],
    delay_seconds=PYTORCH_FETCH["delay_seconds"],
    max_code_examples=PYTORCH_FETCH["max_code_examples"],
)

pprint(pytorch_fetch_summary)

### 4.1 Inspect PyTorch Docs Dataset

In [ ]:
if not PYTORCH_DOCS_FILE.exists():
    print(f"File not found: {PYTORCH_DOCS_FILE}")
else:
    with open(PYTORCH_DOCS_FILE, encoding="utf-8") as file_handle:
        pytorch_docs = json.load(file_handle)

    print(f"Total pages: {len(pytorch_docs)}")
    if pytorch_docs:
        first_page = pytorch_docs[0]
        pprint(
            {
                "title": first_page["title"],
                "url": first_page["url"],
                "content_chars": len(first_page["content"]),
                "code_examples": len(first_page["code_examples"]),
            }
        )
    else:
        print("Dataset is empty.")

## 5. Bootstrap New Collection — ArXiv

Use this cell only when the target alias does not exist yet and you need the first production collection.

In [ ]:
ARXIV_BOOTSTRAP = {
    "alias": "champion",
    "collection_name": None,
    "embedding_model": settings.embedding_model,
    "chunking_strategy": "fixed_token",
    "chunk_size": 512,
    "chunk_overlap": 64,
    "sparse_encoder": None,
    "retrieval_capability": "dense",
}

arxiv_bootstrap_result = create_arxiv_collection(
    build_config=BuildConfig(
        chunking_strategy=ARXIV_BOOTSTRAP["chunking_strategy"],
        chunk_size=ARXIV_BOOTSTRAP["chunk_size"],
        chunk_overlap=ARXIV_BOOTSTRAP["chunk_overlap"],
        embedding_model=ARXIV_BOOTSTRAP["embedding_model"],
        sparse_encoder=ARXIV_BOOTSTRAP["sparse_encoder"],
        retrieval_capability=ARXIV_BOOTSTRAP["retrieval_capability"],
    ),
    arxiv_file=str(ARXIV_DATA_FILE),
    kb="arxiv",
    alias=ARXIV_BOOTSTRAP["alias"],
    collection_name=ARXIV_BOOTSTRAP["collection_name"],
    implementation=ImplementationInfo(module="rag.ops.create.arxiv", experimental=False),
)

pprint(arxiv_bootstrap_result)

## 5.1 Bootstrap New Collection — PyTorch Docs

In [ ]:
PYTORCH_BOOTSTRAP = {
    "alias": "champion",
    "collection_name": None,
    "embedding_model": settings.embedding_model,
    "chunking_strategy": "code",
    "chunk_size": 512,
    "chunk_overlap": 64,
    "sparse_encoder": None,
    "retrieval_capability": "dense",
}

pytorch_bootstrap_result = create_pytorch_docs_collection(
    build_config=BuildConfig(
        chunking_strategy=PYTORCH_BOOTSTRAP["chunking_strategy"],
        chunk_size=PYTORCH_BOOTSTRAP["chunk_size"],
        chunk_overlap=PYTORCH_BOOTSTRAP["chunk_overlap"],
        embedding_model=PYTORCH_BOOTSTRAP["embedding_model"],
        sparse_encoder=PYTORCH_BOOTSTRAP["sparse_encoder"],
        retrieval_capability=PYTORCH_BOOTSTRAP["retrieval_capability"],
    ),
    pytorch_docs_file=str(PYTORCH_DOCS_FILE),
    kb="pytorch_docs",
    alias=PYTORCH_BOOTSTRAP["alias"],
    collection_name=PYTORCH_BOOTSTRAP["collection_name"],
    implementation=ImplementationInfo(module="rag.ops.create.pytorch_docs", experimental=False),
)

pprint(pytorch_bootstrap_result)

## 6. Refresh Existing Alias — ArXiv

Use this after the first collection already exists and has valid `_meta`. This is the same production-safe update path that the Airflow DAG uses.

In [ ]:
ARXIV_REFRESH_ALIAS = "champion"

arxiv_refresh_result = update_arxiv_collection(
    arxiv_file=str(ARXIV_DATA_FILE),
    kb="arxiv",
    alias=ARXIV_REFRESH_ALIAS,
)

pprint(arxiv_refresh_result)

## 6.1 Refresh Existing Alias — PyTorch Docs

In [ ]:
PYTORCH_REFRESH_ALIAS = "champion"

pytorch_refresh_result = update_pytorch_docs_collection(
    pytorch_docs_file=str(PYTORCH_DOCS_FILE),
    kb="pytorch_docs",
    alias=PYTORCH_REFRESH_ALIAS,
)

pprint(pytorch_refresh_result)

## 7. Alias Management

These cells use the production-safe alias helpers. Each cell below performs exactly one alias operation.

In [ ]:
ASSIGN_KB = "arxiv"
ASSIGN_ALIAS_NAME = "challenger"
ASSIGN_COLLECTION_NAME = "arxiv_YYYYMMDD_HHMMSS"

assign_alias_result = assign_alias_to_collection(
    kb=ASSIGN_KB,
    alias=ASSIGN_ALIAS_NAME,
    collection_name=ASSIGN_COLLECTION_NAME,
)

pprint(assign_alias_result)

In [ ]:
PROMOTE_KB = "arxiv"
PROMOTE_FROM_ALIAS = "challenger"
PROMOTE_TO_ALIAS = "champion"

promote_alias_result = promote_alias(
    kb=PROMOTE_KB,
    from_alias=PROMOTE_FROM_ALIAS,
    to_alias=PROMOTE_TO_ALIAS,
)

pprint(promote_alias_result)

In [ ]:
DETACH_KB = "arxiv"
DETACH_ALIAS_NAME = "challenger"

detach_alias_result = detach_alias(
    kb=DETACH_KB,
    alias=DETACH_ALIAS_NAME,
)

pprint(detach_alias_result)

## 8. Diagnostics

Use these strict `rag.ops` inspection helpers before dropping to raw Qdrant metadata.

In [ ]:
DIAGNOSTIC_KB = "arxiv"
DIAGNOSTIC_ALIAS = "champion"

try:
    pprint(inspect_alias(kb_name=DIAGNOSTIC_KB, alias=DIAGNOSTIC_ALIAS))
except Exception as exc:
    print(f"{type(exc).__name__}: {exc}")
    print("If strict inspection fails, continue with the direct Qdrant exploration section below.")

In [ ]:
DIAGNOSTIC_COLLECTION_NAME = "arxiv_YYYYMMDD_HHMMSS"

try:
    pprint(inspect_collection(collection_name=DIAGNOSTIC_COLLECTION_NAME))
except Exception as exc:
    print(f"{type(exc).__name__}: {exc}")
    print("If strict inspection fails, continue with the direct Qdrant exploration section below.")

## 9. Direct Qdrant Exploration

These cells talk to Qdrant directly instead of going through `rag.ops`.
Use them when you need raw state, alias resolution details, or direct metadata inspection.

In [ ]:
admin_store = QdrantVectorStore(
    host=settings.qdrant_host,
    port=settings.qdrant_port,
    collection_name="_qdrant_admin",
)

print(f"Connected to Qdrant at {settings.qdrant_host}:{settings.qdrant_port}")

### 9.1 List Collections And Aliases

In [ ]:
collection_names = sorted(
    collection.name for collection in admin_store.client.get_collections().collections
)
alias_rows = sorted(admin_store.list_aliases(), key=lambda row: row["alias_name"])

print(f"Collections: {len(collection_names)}")
pprint(collection_names)
print(f"\nAliases: {len(alias_rows)}")
pprint(alias_rows)

### 9.2 Resolve An Alias Or Inspect A Collection

In [ ]:
LOOKUP_NAME = "arxiv_champion"

resolved_collection = admin_store.resolve_alias(LOOKUP_NAME)
target_name = resolved_collection or LOOKUP_NAME
target_store = QdrantVectorStore(
    host=settings.qdrant_host,
    port=settings.qdrant_port,
    collection_name=target_name,
)

pprint(
    {
        "lookup_name": LOOKUP_NAME,
        "resolved_collection": resolved_collection,
        "collection_info": target_store.get_collection_info(),
    }
)

### 9.3 Read Raw `_meta`

In [ ]:
TARGET_NAME = "arxiv_champion"

resolved_collection = admin_store.resolve_alias(TARGET_NAME)
meta_target = resolved_collection or TARGET_NAME
meta_store = QdrantVectorStore(
    host=settings.qdrant_host,
    port=settings.qdrant_port,
    collection_name=meta_target,
)

pprint(
    {
        "lookup_name": TARGET_NAME,
        "resolved_collection": resolved_collection,
        "raw_meta": meta_store.read_meta(),
    }
)

### 9.4 Find Collections Without Aliases

In [ ]:
all_collections = {
    collection.name for collection in admin_store.client.get_collections().collections
}
aliased_collections = {row["collection_name"] for row in admin_store.list_aliases()}
orphan_collections = sorted(all_collections - aliased_collections)

pprint(
    {
        "orphan_collections": orphan_collections,
        "aliased_collections": sorted(aliased_collections),
    }
)

## 10. Raw Qdrant Management

These cells bypass the production-safe validation in `rag.ops`.
Use them only for manual repair or cleanup, and double-check names before running them.

In [ ]:
RAW_ALIAS_NAME = "pytorch_docs_champion_staging"
RAW_COLLECTION_NAME = "pytorch_docs_YYYYMMDD_HHMMSS"

admin_store.update_alias(alias_name=RAW_ALIAS_NAME, collection_name=RAW_COLLECTION_NAME)
pprint({"alias_name": RAW_ALIAS_NAME, "collection_name": RAW_COLLECTION_NAME})

In [ ]:
RAW_ALIAS_NAME_TO_DELETE = "pytorch_docs_champion_staging"
CONFIRM_RAW_ALIAS_DELETE = False

if CONFIRM_RAW_ALIAS_DELETE:
    admin_store.delete_alias(alias_name=RAW_ALIAS_NAME_TO_DELETE)
    print(f"Deleted alias: {RAW_ALIAS_NAME_TO_DELETE}")
else:
    print("Set CONFIRM_RAW_ALIAS_DELETE = True to delete the raw alias after checking the name.")

In [ ]:
DELETE_COLLECTION_NAME = None
CONFIRM_DELETE = False

if DELETE_COLLECTION_NAME:
    preview_store = QdrantVectorStore(
        host=settings.qdrant_host,
        port=settings.qdrant_port,
        collection_name=DELETE_COLLECTION_NAME,
    )
    pprint(
        {
            "collection_name": DELETE_COLLECTION_NAME,
            "collection_info": preview_store.get_collection_info(),
            "raw_meta": preview_store.read_meta(),
        }
    )

if DELETE_COLLECTION_NAME and CONFIRM_DELETE:
    admin_store.delete_collection(DELETE_COLLECTION_NAME)
    print(f"Deleted collection: {DELETE_COLLECTION_NAME}")
else:
    print(
        "Set DELETE_COLLECTION_NAME and CONFIRM_DELETE = True "
        "to delete a collection after previewing it."
    )